# Revenue Leakage Intelligence Report

**Purpose:** Quantify and rank addressable revenue leakage across 7 layers, with predictive risk scoring and ranked interventions.  
**Data:** Olist Brazilian E-Commerce (100K orders, 9 tables)  
**Audience:** Wellness Forever leadership team (POC demo)  
**Built by:** Tasknova Insight Framework

---

### The 7 Layers of Revenue Leakage

| # | Leakage Layer | What It Measures | Pharma Equivalent |
|---|---------------|------------------|--------------------|
| 1 | **Order Failure** | Canceled + unavailable orders | Failed Rx fulfillment, stockouts |
| 2 | **Delivery SLA Breach** | Late deliveries → churn risk | Late medicine = missed doses = permanent switch |
| 3 | **Single-Purchase Drain** | ~97% never reorder | Chronic medication refill gap |
| 4 | **Basket Expansion Gap** | Single-category buyers miss cross-sell | Rx-to-OTC attach rate |
| 5 | **Freight Cost Leakage** | High freight kills conversion | Delivery cost in Tier-2/3 cities |
| 6 | **Seller Performance Drag** | Bottom-quartile sellers destroy value | Underperforming store locations |
| 7 | **Payment Friction** | Boleto abandonment + installment friction | UPI/wallet friction for chronic Rx |

**Goal:** Sum all 7 layers into a single **Total Recoverable Revenue Leakage** number, then rank interventions by R$ impact.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Executive styling (from NB07)
plt.rcParams.update({
    'figure.facecolor': '#fafaf9',
    'axes.facecolor': '#fafaf9',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#e5e5e5',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

COLORS = {
    'primary': '#18181b',
    'gold': '#b8860b',
    'green': '#059669',
    'red': '#dc2626',
    'blue': '#2563eb',
    'muted': '#6b7280',
    'light': '#e5e7eb',
    'orange': '#ea580c',
    'purple': '#7c3aed',
}

# Shared utilities
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'notebooks') if not os.path.exists('insight_utils.py') else '.')
from insight_utils import InsightCollector, format_currency, format_pct, pareto_analysis, detect_anomalies_iqr

insights = InsightCollector()
leakage = {}

def fmt_brl(val):
    """Format value as Brazilian Real."""
    if abs(val) >= 1e6:
        return f"R${val/1e6:.2f}M"
    elif abs(val) >= 1e3:
        return f"R${val/1e3:.1f}K"
    else:
        return f"R${val:.0f}"

print('Setup complete.')

In [ ]:
# Load ALL 9 Olist tables
import kagglehub
path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(f'Dataset path: {path}')

orders    = pd.read_csv(os.path.join(path, 'olist_orders_dataset.csv'))
items     = pd.read_csv(os.path.join(path, 'olist_order_items_dataset.csv'))
products  = pd.read_csv(os.path.join(path, 'olist_products_dataset.csv'))
customers = pd.read_csv(os.path.join(path, 'olist_customers_dataset.csv'))
payments  = pd.read_csv(os.path.join(path, 'olist_order_payments_dataset.csv'))
reviews   = pd.read_csv(os.path.join(path, 'olist_order_reviews_dataset.csv'))
sellers   = pd.read_csv(os.path.join(path, 'olist_sellers_dataset.csv'))
geo       = pd.read_csv(os.path.join(path, 'olist_geolocation_dataset.csv'))
cat_trans = pd.read_csv(os.path.join(path, 'product_category_name_translation.csv'))

# Parse date columns
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print(f'\nTables loaded:')
for name, tbl in [('orders', orders), ('items', items), ('products', products),
                   ('customers', customers), ('payments', payments),
                   ('reviews', reviews), ('sellers', sellers), ('cat_trans', cat_trans)]:
    print(f'  {name:12s}: {tbl.shape[0]:>8,} rows x {tbl.shape[1]} cols')

In [ ]:
# Build master table — ALL order statuses included
df = items.merge(orders, on='order_id', how='left')
df = df.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
df = df.merge(cat_trans, on='product_category_name', how='left')
df = df.merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left')
df = df.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

# Order-level payment aggregation
pay_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_type=('payment_type', 'first'),
    payment_installments=('payment_installments', 'max')
).reset_index()
df = df.merge(pay_agg, on='order_id', how='left')

# Order-level review aggregation
rev_agg = reviews.groupby('order_id')['review_score'].mean().reset_index()
df = df.merge(rev_agg, on='order_id', how='left')

# Computed columns
df['revenue'] = df['price'] + df['freight_value']
df['freight_ratio'] = df['freight_value'] / df['price'].replace(0, np.nan)
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.total_seconds() / 86400
df['is_late'] = df['delivery_delay_days'] > 0
df['category'] = df['product_category_name_english'].fillna(df['product_category_name'])

total_revenue = df['revenue'].sum()

print(f'Master table: {len(df):,} rows x {df.shape[1]} cols')
print(f'Order statuses: {df["order_status"].value_counts().to_dict()}')
print(f'Total revenue (all statuses): {fmt_brl(total_revenue)}')
print(f'Date range: {df["order_purchase_timestamp"].min().date()} to {df["order_purchase_timestamp"].max().date()}')

---
## 1. Order Failure Leakage

Revenue lost from orders that were **canceled** or marked **unavailable** before fulfillment.

In [ ]:
# Order failure analysis
failed_statuses = ['canceled', 'unavailable']
failed = df[df['order_status'].isin(failed_statuses)].copy()
failed_revenue = failed['revenue'].sum()
leakage['order_failure'] = failed_revenue

print(f'Failed orders: {failed["order_id"].nunique():,} ({failed["order_id"].nunique()/df["order_id"].nunique()*100:.1f}%)')
print(f'Revenue lost: {fmt_brl(failed_revenue)} ({failed_revenue/total_revenue*100:.2f}% of total)')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. By status
status_rev = failed.groupby('order_status')['revenue'].sum().sort_values(ascending=False)
status_rev.plot(kind='bar', ax=axes[0, 0], color=[COLORS['red'], COLORS['orange']], edgecolor='white')
axes[0, 0].set_title('Failed Revenue by Status', fontweight='bold')
axes[0, 0].set_ylabel('Revenue (R$)')
axes[0, 0].tick_params(axis='x', rotation=0)

# 2. Top categories by failure
cat_fail = failed.groupby('category')['revenue'].sum().sort_values(ascending=False).head(10)
cat_fail.plot(kind='barh', ax=axes[0, 1], color=COLORS['red'], edgecolor='white')
axes[0, 1].set_title('Top 10 Categories by Failed Revenue', fontweight='bold')
axes[0, 1].set_xlabel('Revenue (R$)')
axes[0, 1].invert_yaxis()

# 3. By state
state_fail = failed.groupby('customer_state')['revenue'].sum().sort_values(ascending=False).head(10)
state_fail.plot(kind='barh', ax=axes[1, 0], color=COLORS['orange'], edgecolor='white')
axes[1, 0].set_title('Top 10 States by Failed Revenue', fontweight='bold')
axes[1, 0].set_xlabel('Revenue (R$)')
axes[1, 0].invert_yaxis()

# 4. By payment type
pay_fail = failed.groupby('payment_type')['revenue'].sum().sort_values(ascending=False)
pay_fail.plot(kind='bar', ax=axes[1, 1], color=COLORS['gold'], edgecolor='white')
axes[1, 1].set_title('Failed Revenue by Payment Type', fontweight='bold')
axes[1, 1].set_ylabel('Revenue (R$)')
axes[1, 1].tick_params(axis='x', rotation=30)

plt.suptitle(f'Layer 1: Order Failure Leakage = {fmt_brl(failed_revenue)}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

insights.add('Order Failure', 'Canceled & Unavailable Orders',
             f'{fmt_brl(failed_revenue)} lost from {failed["order_id"].nunique():,} failed orders. '
             f'Top category: {cat_fail.index[0]}. Top state: {state_fail.index[0]}.',
             severity='risk')

> **Pharma Translation:** Order failures map directly to **failed prescription fulfillment** and **stockouts**. In pharmacy e-commerce, a failed order means a patient doesn't get their medication on time — they switch to a competitor *permanently*. Wellness Forever should track fill-rate by SKU and flag any drug dropping below 98% fulfillment.

---
## 2. Delivery SLA Breach Chain

Late deliveries erode trust, generate bad reviews, and drive churn. We quantify the **revenue at risk** from SLA breaches.

In [ ]:
# SLA breach analysis — delivered orders only
delivered = df[df['order_status'] == 'delivered'].copy()
delivered_with_delay = delivered.dropna(subset=['delivery_delay_days'])

late = delivered_with_delay[delivered_with_delay['is_late']]
late_revenue = late['revenue'].sum()
churn_rate_estimate = 0.15  # ~15% of late-delivery customers churn
sla_leakage = late_revenue * churn_rate_estimate
leakage['sla_breach'] = sla_leakage

print(f'Delivered orders with delay data: {len(delivered_with_delay):,}')
print(f'Late deliveries: {len(late):,} ({len(late)/len(delivered_with_delay)*100:.1f}%)')
print(f'Revenue from late orders: {fmt_brl(late_revenue)}')
print(f'Estimated churn leakage (15% of late revenue): {fmt_brl(sla_leakage)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Delay distribution
delay_data = delivered_with_delay['delivery_delay_days'].clip(-30, 60)
axes[0].hist(delay_data, bins=60, color=COLORS['blue'], edgecolor='white', alpha=0.8)
axes[0].axvline(0, color=COLORS['red'], linestyle='--', linewidth=2, label='SLA deadline')
axes[0].set_title('Delivery Delay Distribution (days)', fontweight='bold')
axes[0].set_xlabel('Days (negative = early, positive = late)')
axes[0].set_ylabel('Order count')
axes[0].legend(framealpha=0)

# Late orders by delay bucket
late_copy = late.copy()
bins = [0, 3, 7, 14, 30, 999]
labels = ['1-3d', '4-7d', '8-14d', '15-30d', '30d+']
late_copy['delay_bucket'] = pd.cut(late_copy['delivery_delay_days'], bins=bins, labels=labels)
bucket_rev = late_copy.groupby('delay_bucket', observed=True)['revenue'].sum()
bucket_rev.plot(kind='bar', ax=axes[1], color=COLORS['red'], edgecolor='white')
axes[1].set_title('Revenue at Risk by Delay Severity', fontweight='bold')
axes[1].set_ylabel('Revenue (R$)')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Layer 2: SLA Breach Leakage = {fmt_brl(sla_leakage)}', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

insights.add('SLA Breach', 'Late Delivery Churn Risk',
             f'{len(late):,} late deliveries put {fmt_brl(late_revenue)} at risk. '
             f'Estimated {fmt_brl(sla_leakage)} in churn leakage (15% rate).',
             severity='risk')

In [ ]:
# Late delivery → bad review correlation
review_delay = delivered_with_delay.dropna(subset=['review_score']).copy()
bins_corr = [-999, -7, 0, 3, 7, 14, 999]
labels_corr = ['7+ early', '1-7 early', '0-3 late', '4-7 late', '8-14 late', '14+ late']
review_delay['delay_bucket'] = pd.cut(review_delay['delivery_delay_days'], bins=bins_corr, labels=labels_corr)

bucket_review = review_delay.groupby('delay_bucket', observed=True).agg(
    mean_score=('review_score', 'mean'),
    pct_1star=('review_score', lambda x: (x <= 1).mean() * 100),
    count=('review_score', 'count')
)

# Spearman correlation
valid = review_delay.dropna(subset=['delivery_delay_days', 'review_score'])
rho, p_val = stats.spearmanr(valid['delivery_delay_days'], valid['review_score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean review by delay bucket
bucket_review['mean_score'].plot(kind='bar', ax=axes[0], color=COLORS['gold'], edgecolor='white')
axes[0].set_title('Mean Review Score by Delivery Timing', fontweight='bold')
axes[0].set_ylabel('Mean review score')
axes[0].set_ylim(1, 5.2)
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(review_delay['review_score'].mean(), color=COLORS['muted'], linestyle='--', alpha=0.5)

# % 1-star reviews
bucket_review['pct_1star'].plot(kind='bar', ax=axes[1], color=COLORS['red'], edgecolor='white')
axes[1].set_title('% 1-Star Reviews by Delivery Timing', fontweight='bold')
axes[1].set_ylabel('% 1-star reviews')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(f'Delivery Delay → Review Impact (Spearman ρ = {rho:.3f}, p < {p_val:.1e})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Spearman correlation (delay → review): ρ = {rho:.3f} (p = {p_val:.2e})')
print(f'Late orders are {bucket_review.loc["14+ late", "pct_1star"]/bucket_review.loc["7+ early", "pct_1star"]:.1f}x more likely to get 1-star reviews')

insights.add('SLA Breach', 'Late Delivery → Bad Reviews',
             f'Spearman ρ={rho:.3f} (p<0.001). Late orders generate '
             f'{bucket_review.loc["14+ late", "pct_1star"]:.0f}% 1-star reviews vs '
             f'{bucket_review.loc["7+ early", "pct_1star"]:.0f}% for early deliveries.',
             severity='risk')

> **Pharma Translation:** Late medicine delivery = **missed doses**. For chronic conditions (diabetes, hypertension), even a 2-day gap can trigger a permanent switch to the nearest pharmacy. Wellness Forever should implement **guaranteed same-day delivery** for chronic Rx refills and track the SLA breach → churn chain as a leading indicator.

---
## 3. Single-Purchase Revenue Drain

~97% of Olist customers never return. We estimate the **LTV gap** if retention improved — the core metric for pharma chronic medication refills.

In [ ]:
# Repeat purchase analysis
delivered_df = df[df['order_status'] == 'delivered'].copy()
cust_orders = delivered_df.groupby('customer_unique_id').agg(
    n_orders=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum'),
    first_order=('order_purchase_timestamp', 'min'),
    last_order=('order_purchase_timestamp', 'max'),
).reset_index()

repeat_customers = cust_orders[cust_orders['n_orders'] > 1]
single_customers = cust_orders[cust_orders['n_orders'] == 1]
repeat_rate = len(repeat_customers) / len(cust_orders) * 100

print(f'Total customers: {len(cust_orders):,}')
print(f'Repeat customers: {len(repeat_customers):,} ({repeat_rate:.1f}%)')
print(f'Single-purchase: {len(single_customers):,} ({100-repeat_rate:.1f}%)')

# Segment single-purchase customers
single_detail = delivered_df.merge(
    single_customers[['customer_unique_id']], on='customer_unique_id'
)

# Replenishable categories (would expect repeat in pharma)
replenishable = ['health_beauty', 'perfumery', 'baby', 'food_drink', 'food',
                 'diapers_and_hygiene', 'drinks']
single_detail['is_replenishable'] = single_detail['category'].isin(replenishable)
single_detail['delivery_experience'] = np.where(single_detail['is_late'] == True, 'Late', 'On-time')
single_detail['review_bucket'] = pd.cut(single_detail['review_score'], bins=[0, 2, 3, 5],
                                         labels=['Poor (1-2)', 'Medium (3)', 'Good (4-5)'])
single_detail['value_bucket'] = pd.qcut(single_detail['revenue'], 3, labels=['Low', 'Medium', 'High'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Repeat rate by replenishable vs durable
replen_rate = delivered_df.copy()
replen_rate['is_replenishable'] = replen_rate['category'].isin(replenishable)
replen_cust = replen_rate.groupby(['customer_unique_id', 'is_replenishable'])['order_id'].nunique().reset_index()
replen_cust['is_repeat'] = replen_cust['order_id'] > 1
rr_by_type = replen_cust.groupby('is_replenishable')['is_repeat'].mean() * 100
rr_by_type.index = ['Durable', 'Replenishable']
rr_by_type.plot(kind='bar', ax=axes[0, 0], color=[COLORS['muted'], COLORS['green']], edgecolor='white')
axes[0, 0].set_title('Repeat Rate: Replenishable vs Durable', fontweight='bold')
axes[0, 0].set_ylabel('Repeat Rate (%)')
axes[0, 0].tick_params(axis='x', rotation=0)

# Repeat rate by delivery experience
del_cust = delivered_df.dropna(subset=['is_late']).groupby(['customer_unique_id']).agg(
    n_orders=('order_id', 'nunique'),
    any_late=('is_late', 'max')
).reset_index()
del_cust['is_repeat'] = del_cust['n_orders'] > 1
del_cust['experience'] = np.where(del_cust['any_late'], 'Had Late Delivery', 'Always On-Time')
rr_by_del = del_cust.groupby('experience')['is_repeat'].mean() * 100
rr_by_del.plot(kind='bar', ax=axes[0, 1], color=[COLORS['green'], COLORS['red']], edgecolor='white')
axes[0, 1].set_title('Repeat Rate by Delivery Experience', fontweight='bold')
axes[0, 1].set_ylabel('Repeat Rate (%)')
axes[0, 1].tick_params(axis='x', rotation=0)

# Repeat rate by review score
rev_cust = delivered_df.dropna(subset=['review_score']).groupby(['customer_unique_id']).agg(
    n_orders=('order_id', 'nunique'),
    avg_review=('review_score', 'mean')
).reset_index()
rev_cust['is_repeat'] = rev_cust['n_orders'] > 1
rev_cust['review_bucket'] = pd.cut(rev_cust['avg_review'], bins=[0, 2, 3, 4, 5],
                                    labels=['1-2', '3', '4', '5'])
rr_by_rev = rev_cust.groupby('review_bucket', observed=True)['is_repeat'].mean() * 100
rr_by_rev.plot(kind='bar', ax=axes[1, 0], color=COLORS['gold'], edgecolor='white')
axes[1, 0].set_title('Repeat Rate by Review Score', fontweight='bold')
axes[1, 0].set_ylabel('Repeat Rate (%)')
axes[1, 0].tick_params(axis='x', rotation=0)

# Repeat rate by order value
val_cust = delivered_df.groupby(['customer_unique_id']).agg(
    n_orders=('order_id', 'nunique'),
    avg_value=('revenue', 'mean')
).reset_index()
val_cust['is_repeat'] = val_cust['n_orders'] > 1
val_cust['value_bucket'] = pd.qcut(val_cust['avg_value'], 4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
rr_by_val = val_cust.groupby('value_bucket', observed=True)['is_repeat'].mean() * 100
rr_by_val.plot(kind='bar', ax=axes[1, 1], color=COLORS['blue'], edgecolor='white')
axes[1, 1].set_title('Repeat Rate by Order Value Quartile', fontweight='bold')
axes[1, 1].set_ylabel('Repeat Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Single-Purchase Analysis (Current repeat rate: {repeat_rate:.1f}%)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# LTV simulation: what if repeat rate improved?
avg_second_order_value = repeat_customers['total_revenue'].mean() / repeat_customers['n_orders'].mean()
n_single = len(single_customers)
current_repeat_pct = repeat_rate / 100

scenarios = [
    ('10% repeat rate', 0.10),
    ('15% repeat rate', 0.15),
    ('20% repeat rate', 0.20),
]

print(f'Avg second-order value (from existing repeaters): {fmt_brl(avg_second_order_value)}')
print(f'Single-purchase customers: {n_single:,}')
print(f'\nLTV SIMULATION:')
print(f'{"Scenario":<25s} {"New Repeaters":>14s} {"Incremental Rev":>16s}')
print('-' * 60)

scenario_data = []
for label, target_rate in scenarios:
    new_repeaters = int(len(cust_orders) * (target_rate - current_repeat_pct))
    incremental = new_repeaters * avg_second_order_value
    scenario_data.append({'scenario': label, 'new_repeaters': new_repeaters, 'incremental': incremental})
    print(f'{label:<25s} {new_repeaters:>14,} {fmt_brl(incremental):>16s}')

# Use 15% scenario as the leakage estimate
leakage['repeat_gap'] = scenario_data[1]['incremental']
print(f'\nStored leakage (15% scenario): {fmt_brl(leakage["repeat_gap"])}')

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([s['scenario'] for s in scenario_data],
              [s['incremental'] for s in scenario_data],
              color=[COLORS['green'], COLORS['gold'], COLORS['blue']], edgecolor='white', width=0.5)
for bar, s in zip(bars, scenario_data):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            fmt_brl(s['incremental']), ha='center', fontweight='bold', fontsize=12)
ax.set_title(f'Layer 3: Repeat Gap Leakage = {fmt_brl(leakage["repeat_gap"])} (15% scenario)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Incremental Revenue (R$)')
plt.tight_layout()
plt.show()

insights.add('Repeat Gap', 'Single-Purchase Revenue Drain',
             f'Only {repeat_rate:.1f}% of customers reorder. Lifting to 15% would recover '
             f'{fmt_brl(leakage["repeat_gap"])} in incremental revenue.',
             severity='opportunity')

> **Pharma Translation:** This is the **chronic medication refill gap** — the single biggest leakage for pharmacy chains. A diabetes patient who fills their first prescription but never returns represents 12+ months of lost refill revenue. **Auto-refill programs** with SMS/WhatsApp reminders are the highest-ROI intervention Wellness Forever can implement. The 15% scenario is conservative for pharma — chronic Rx refill rates should target 60-80%.

---
## 4. Basket Expansion Gap

Single-category buyers leave cross-sell revenue on the table.

In [ ]:
# Basket expansion analysis
cust_cats = delivered_df.groupby('customer_unique_id').agg(
    n_categories=('category', 'nunique'),
    total_revenue=('revenue', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()

single_cat = cust_cats[cust_cats['n_categories'] == 1]
multi_cat = cust_cats[cust_cats['n_categories'] > 1]

single_cat_aov = single_cat['total_revenue'].mean()
multi_cat_aov = multi_cat['total_revenue'].mean()
basket_gap = len(single_cat) * (multi_cat_aov - single_cat_aov)
# Conservative: only 20% of single-cat customers could realistically expand
basket_gap_conservative = basket_gap * 0.20
leakage['basket_gap'] = basket_gap_conservative

print(f'Single-category customers: {len(single_cat):,} ({len(single_cat)/len(cust_cats)*100:.1f}%)')
print(f'Multi-category customers: {len(multi_cat):,} ({len(multi_cat)/len(cust_cats)*100:.1f}%)')
print(f'Avg revenue — single-cat: {fmt_brl(single_cat_aov)} | multi-cat: {fmt_brl(multi_cat_aov)}')
print(f'Revenue uplift per customer if expanded: {fmt_brl(multi_cat_aov - single_cat_aov)}')
print(f'Total gap (20% conversion): {fmt_brl(basket_gap_conservative)}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AOV comparison
aov_data = pd.Series({'Single-Category': single_cat_aov, 'Multi-Category': multi_cat_aov})
aov_data.plot(kind='bar', ax=axes[0], color=[COLORS['muted'], COLORS['green']], edgecolor='white', width=0.5)
axes[0].set_title('Avg Customer Revenue: Single vs Multi-Category', fontweight='bold')
axes[0].set_ylabel('Avg Revenue per Customer (R$)')
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(aov_data):
    axes[0].text(i, v + 2, fmt_brl(v), ha='center', fontweight='bold')

# Distribution of categories per customer
cat_dist = cust_cats['n_categories'].value_counts().sort_index().head(8)
cat_dist.plot(kind='bar', ax=axes[1], color=COLORS['blue'], edgecolor='white')
axes[1].set_title('Number of Categories per Customer', fontweight='bold')
axes[1].set_xlabel('Categories purchased')
axes[1].set_ylabel('Customer count')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Layer 4: Basket Expansion Gap = {fmt_brl(basket_gap_conservative)}',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

insights.add('Basket Gap', 'Cross-Sell Opportunity',
             f'{len(single_cat):,} single-category customers. Multi-cat customers spend '
             f'{fmt_brl(multi_cat_aov - single_cat_aov)} more. Conservative gap: {fmt_brl(basket_gap_conservative)}.',
             severity='opportunity')

> **Pharma Translation:** This is the **Rx-to-OTC attach rate**. A patient filling a blood pressure prescription should also be prompted for a glucometer, supplements, or personal care items. Pharmacist-prompted cross-sells at the counter convert at 15-25%. Wellness Forever should measure attach rate per pharmacist and per store as a KPI.

---
## 5. Freight Cost Leakage

When shipping costs are a large fraction of product price, customers abandon or downgrade orders.

In [ ]:
# Freight cost leakage
freight_df = delivered_df.dropna(subset=['freight_ratio']).copy()
freight_df = freight_df[freight_df['freight_ratio'] < 10]  # remove extreme outliers

# Completion rate by freight ratio bucket
all_orders = df.dropna(subset=['freight_ratio']).copy()
all_orders = all_orders[all_orders['freight_ratio'] < 10]
all_orders['freight_bucket'] = pd.cut(all_orders['freight_ratio'],
                                       bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0, 10],
                                       labels=['<10%', '10-20%', '20-30%', '30-50%', '50-100%', '>100%'])
all_orders['is_completed'] = all_orders['order_status'] == 'delivered'

freight_completion = all_orders.groupby('freight_bucket', observed=True).agg(
    completion_rate=('is_completed', 'mean'),
    count=('order_id', 'count'),
    total_revenue=('revenue', 'sum')
)
freight_completion['completion_rate'] *= 100

# High-freight orders (>30% ratio) that completed — estimate lost margin
high_freight = freight_df[freight_df['freight_ratio'] > 0.30]
excess_freight = (high_freight['freight_value'] - high_freight['price'] * 0.30).clip(lower=0).sum()
leakage['freight'] = excess_freight

# High-freight zones by state
state_freight = freight_df.groupby('customer_state')['freight_ratio'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Completion rate by freight bucket
x = range(len(freight_completion))
ax1 = axes[0]
bars = ax1.bar(x, freight_completion['completion_rate'],
               color=[COLORS['green'] if r > 97 else COLORS['gold'] if r > 95 else COLORS['red']
                      for r in freight_completion['completion_rate']],
               edgecolor='white', width=0.6)
ax1.set_xticks(x)
ax1.set_xticklabels(freight_completion.index)
ax1.set_title('Order Completion Rate by Freight-to-Price Ratio', fontweight='bold')
ax1.set_ylabel('Completion Rate (%)')
ax1.set_xlabel('Freight / Price Ratio')
ax1.set_ylim(90, 101)
ax1.axhline(97, color=COLORS['muted'], linestyle='--', alpha=0.5)
for bar, rate in zip(bars, freight_completion['completion_rate']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{rate:.1f}%', ha='center', fontsize=10)

# High-freight states
state_freight.head(15).sort_values().plot(kind='barh', ax=axes[1], color=COLORS['orange'], edgecolor='white')
axes[1].set_title('Average Freight Ratio by State (Top 15 Highest)', fontweight='bold')
axes[1].set_xlabel('Avg Freight / Price Ratio')
axes[1].axvline(0.30, color=COLORS['red'], linestyle='--', linewidth=1.5, label='30% threshold')
axes[1].legend(framealpha=0)

plt.suptitle(f'Layer 5: Freight Cost Leakage = {fmt_brl(excess_freight)}',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Orders with freight > 30% of price: {len(high_freight):,} ({len(high_freight)/len(freight_df)*100:.1f}%)')
print(f'Excess freight cost above 30% threshold: {fmt_brl(excess_freight)}')
print(f'Highest freight states: {state_freight.head(3).index.tolist()}')

insights.add('Freight Leakage', 'High Shipping Cost Drag',
             f'{len(high_freight):,} orders have freight >30% of price. '
             f'Excess freight cost: {fmt_brl(excess_freight)}. Worst states: {state_freight.head(3).index.tolist()}.',
             severity='risk')

> **Pharma Translation:** Delivery cost in **Tier-2 and Tier-3 Indian cities** is the equivalent problem. Wellness Forever should consider **dark stores** (mini-fulfillment centers) in high-demand areas where last-mile cost exceeds 30% of order value. Subscription-based free delivery for chronic Rx patients would also reduce the freight friction.

---
## 6. Seller Performance Drag

Bottom-quartile sellers generate complaints, returns, and churn. Revenue flowing through underperformers is at risk.

In [ ]:
# Seller performance analysis
seller_metrics = delivered_df.groupby('seller_id').agg(
    n_orders=('order_id', 'nunique'),
    revenue=('revenue', 'sum'),
    avg_review=('review_score', 'mean'),
    avg_delay=('delivery_delay_days', 'mean'),
    late_pct=('is_late', 'mean'),
).reset_index()

# Add cancellation rate from all orders
seller_cancel = df.groupby('seller_id').apply(
    lambda x: (x['order_status'].isin(['canceled', 'unavailable'])).mean()
).reset_index(name='cancel_rate')
seller_metrics = seller_metrics.merge(seller_cancel, on='seller_id', how='left')

# Filter to sellers with enough orders for reliable stats
seller_metrics = seller_metrics[seller_metrics['n_orders'] >= 5].copy()

# Composite score (higher = better)
seller_metrics['review_norm'] = (seller_metrics['avg_review'] - seller_metrics['avg_review'].min()) / \
                                 (seller_metrics['avg_review'].max() - seller_metrics['avg_review'].min())
seller_metrics['delay_norm'] = 1 - (seller_metrics['avg_delay'].clip(-30, 30) - seller_metrics['avg_delay'].clip(-30, 30).min()) / \
                                    (seller_metrics['avg_delay'].clip(-30, 30).max() - seller_metrics['avg_delay'].clip(-30, 30).min())
seller_metrics['cancel_norm'] = 1 - (seller_metrics['cancel_rate'] - seller_metrics['cancel_rate'].min()) / \
                                     (seller_metrics['cancel_rate'].max() - seller_metrics['cancel_rate'].min() + 1e-9)

seller_metrics['composite_score'] = (
    seller_metrics['review_norm'] * 0.4 +
    seller_metrics['delay_norm'] * 0.4 +
    seller_metrics['cancel_norm'] * 0.2
)

# Quartile ranking
seller_metrics['quartile'] = pd.qcut(seller_metrics['composite_score'], 4,
                                      labels=['Q1 (Bottom)', 'Q2', 'Q3', 'Q4 (Top)'])

# Revenue through bottom-quartile sellers
bottom_q = seller_metrics[seller_metrics['quartile'] == 'Q1 (Bottom)']
bottom_q_revenue = bottom_q['revenue'].sum()
seller_drag = bottom_q_revenue * 0.12  # 10-15% risk estimate
leakage['seller_drag'] = seller_drag

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Scatter: review vs delay colored by quartile
colors_q = {'Q1 (Bottom)': COLORS['red'], 'Q2': COLORS['orange'],
            'Q3': COLORS['gold'], 'Q4 (Top)': COLORS['green']}
for q, grp in seller_metrics.groupby('quartile', observed=True):
    axes[0, 0].scatter(grp['avg_delay'], grp['avg_review'], s=grp['revenue']/500,
                       c=colors_q[q], alpha=0.5, label=q, edgecolors='white', linewidth=0.5)
axes[0, 0].set_xlabel('Avg Delivery Delay (days)')
axes[0, 0].set_ylabel('Avg Review Score')
axes[0, 0].set_title('Seller Quality: Review vs Delay', fontweight='bold')
axes[0, 0].legend(fontsize=9, framealpha=0)

# Revenue by quartile
q_rev = seller_metrics.groupby('quartile', observed=True)['revenue'].sum()
q_rev.plot(kind='bar', ax=axes[0, 1],
           color=[colors_q[q] for q in q_rev.index], edgecolor='white')
axes[0, 1].set_title('Revenue by Seller Quartile', fontweight='bold')
axes[0, 1].set_ylabel('Revenue (R$)')
axes[0, 1].tick_params(axis='x', rotation=0)

# Metrics by quartile
q_metrics = seller_metrics.groupby('quartile', observed=True).agg(
    avg_review=('avg_review', 'mean'),
    avg_delay=('avg_delay', 'mean'),
    cancel_rate=('cancel_rate', 'mean'),
    n_sellers=('seller_id', 'count')
)
q_metrics['avg_review'].plot(kind='bar', ax=axes[1, 0],
                              color=[colors_q[q] for q in q_metrics.index], edgecolor='white')
axes[1, 0].set_title('Avg Review Score by Quartile', fontweight='bold')
axes[1, 0].set_ylabel('Avg Review')
axes[1, 0].set_ylim(1, 5.2)
axes[1, 0].tick_params(axis='x', rotation=0)

# Cancellation rate by quartile
(q_metrics['cancel_rate'] * 100).plot(kind='bar', ax=axes[1, 1],
                                       color=[colors_q[q] for q in q_metrics.index], edgecolor='white')
axes[1, 1].set_title('Cancel Rate by Quartile', fontweight='bold')
axes[1, 1].set_ylabel('Cancel Rate (%)')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Layer 6: Seller Drag Leakage = {fmt_brl(seller_drag)}',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Sellers analyzed (5+ orders): {len(seller_metrics):,}')
print(f'Bottom-quartile sellers: {len(bottom_q):,}')
print(f'Revenue through bottom-Q: {fmt_brl(bottom_q_revenue)} ({bottom_q_revenue/delivered_df["revenue"].sum()*100:.1f}% of delivered revenue)')
print(f'Estimated drag leakage (12% risk): {fmt_brl(seller_drag)}')

insights.add('Seller Drag', 'Bottom-Quartile Seller Risk',
             f'{len(bottom_q):,} bottom-quartile sellers handle {fmt_brl(bottom_q_revenue)} in revenue. '
             f'Estimated {fmt_brl(seller_drag)} at risk from poor experience.',
             severity='risk')

> **Pharma Translation:** Bottom-quartile sellers map to **underperforming store locations**. Wellness Forever should rank stores by composite score (fill rate, delivery time, customer complaints) and deploy ops intervention teams to the bottom quartile. A 12% revenue risk from poor store experience is conservative — pharma has higher switching costs but also higher lifetime value per customer.

---
## 7. Payment Friction

Boleto (bank slip) payments have inherently higher abandonment. Installment complexity adds friction.

In [ ]:
# Payment friction analysis
pay_df = df.copy()
pay_df['is_completed'] = pay_df['order_status'] == 'delivered'

# Failure rate by payment type
pay_type_rates = pay_df.groupby('payment_type').agg(
    completion_rate=('is_completed', 'mean'),
    n_orders=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum')
)
pay_type_rates['failure_rate'] = (1 - pay_type_rates['completion_rate']) * 100
pay_type_rates = pay_type_rates[pay_type_rates['n_orders'] >= 50]  # significant sample

# Failure rate by installment bucket
inst_df = pay_df[pay_df['payment_type'] == 'credit_card'].copy()
inst_df['installment_bucket'] = pd.cut(inst_df['payment_installments'],
                                        bins=[0, 1, 3, 6, 12, 24],
                                        labels=['1x', '2-3x', '4-6x', '7-12x', '13-24x'])
inst_rates = inst_df.groupby('installment_bucket', observed=True).agg(
    completion_rate=('is_completed', 'mean'),
    n_orders=('order_id', 'nunique')
)
inst_rates['failure_rate'] = (1 - inst_rates['completion_rate']) * 100

# Boleto abandonment premium
boleto = pay_type_rates.loc['boleto'] if 'boleto' in pay_type_rates.index else None
credit = pay_type_rates.loc['credit_card'] if 'credit_card' in pay_type_rates.index else None

if boleto is not None and credit is not None:
    boleto_premium = (boleto['failure_rate'] - credit['failure_rate']) / 100
    boleto_lost = boleto['total_revenue'] * boleto_premium
else:
    boleto_lost = 0

leakage['payment_friction'] = max(boleto_lost, 0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Failure rate by payment type
pay_type_rates['failure_rate'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color=COLORS['red'], edgecolor='white')
axes[0].set_title('Order Failure Rate by Payment Type', fontweight='bold')
axes[0].set_ylabel('Failure Rate (%)')
axes[0].tick_params(axis='x', rotation=30)

# Failure rate by installments
inst_rates['failure_rate'].plot(kind='bar', ax=axes[1], color=COLORS['purple'], edgecolor='white')
axes[1].set_title('Failure Rate by Installment Count (Credit Card)', fontweight='bold')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle(f'Layer 7: Payment Friction Leakage = {fmt_brl(leakage["payment_friction"])}',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nPayment Type Failure Rates:')
print(pay_type_rates[['n_orders', 'failure_rate']].sort_values('failure_rate', ascending=False).to_string())
if boleto is not None and credit is not None:
    print(f'\nBoleto abandonment premium vs credit card: {boleto_premium*100:.2f}pp')
    print(f'Estimated boleto leakage: {fmt_brl(leakage["payment_friction"])}')

insights.add('Payment Friction', 'Boleto & Installment Friction',
             f'Boleto failure rate is {boleto["failure_rate"]:.1f}% vs credit card {credit["failure_rate"]:.1f}%. '
             f'Estimated leakage: {fmt_brl(leakage["payment_friction"])}.',
             severity='risk')

> **Pharma Translation:** In India, **UPI/wallet friction** is the equivalent. Wellness Forever should implement **one-tap reorder** for chronic Rx patients — saved payment method + saved address + saved prescription = zero-friction refill. Every additional tap in the checkout flow costs 5-10% conversion for repeat medication orders.

---
## 8. Unified Revenue Leakage Summary

All 7 layers combined into a single **Total Recoverable Revenue Leakage** number.

In [ ]:
# Headline number + waterfall chart
layer_names = [
    'Order Failure',
    'SLA Breach',
    'Repeat Gap',
    'Basket Gap',
    'Freight Cost',
    'Seller Drag',
    'Payment Friction',
]
layer_keys = ['order_failure', 'sla_breach', 'repeat_gap', 'basket_gap',
              'freight', 'seller_drag', 'payment_friction']
layer_values = [leakage.get(k, 0) for k in layer_keys]
total_leakage = sum(layer_values)

print(f'={"=" * 60}')
print(f'  TOTAL RECOVERABLE REVENUE LEAKAGE: {fmt_brl(total_leakage)}')
print(f'  ({total_leakage/total_revenue*100:.1f}% of total revenue {fmt_brl(total_revenue)})')
print(f'={"=" * 60}')

# Waterfall chart
fig, ax = plt.subplots(figsize=(14, 7))

# Sort layers by value descending for impact
sorted_idx = np.argsort(layer_values)[::-1]
sorted_names = [layer_names[i] for i in sorted_idx]
sorted_values = [layer_values[i] for i in sorted_idx]

# Cumulative waterfall
cumulative = 0
bar_colors = [COLORS['red'], COLORS['orange'], COLORS['gold'], COLORS['blue'],
              COLORS['purple'], COLORS['muted'], COLORS['green']]

positions = list(range(len(sorted_names) + 1))
for i, (name, val) in enumerate(zip(sorted_names, sorted_values)):
    ax.bar(i, val, bottom=cumulative, color=bar_colors[i % len(bar_colors)],
           edgecolor='white', width=0.6)
    ax.text(i, cumulative + val/2, fmt_brl(val), ha='center', va='center',
            fontweight='bold', fontsize=10, color='white')
    cumulative += val

# Total bar
ax.bar(len(sorted_names), total_leakage, color=COLORS['primary'], edgecolor='white', width=0.6)
ax.text(len(sorted_names), total_leakage/2, fmt_brl(total_leakage), ha='center', va='center',
        fontweight='bold', fontsize=12, color='white')

ax.set_xticks(positions)
ax.set_xticklabels(sorted_names + ['TOTAL'], rotation=30, ha='right', fontsize=11)
ax.set_ylabel('Revenue Leakage (R$)', fontsize=12)
ax.set_title(f'Total Recoverable Revenue Leakage: {fmt_brl(total_leakage)} '
             f'({total_leakage/total_revenue*100:.1f}% of Revenue)',
             fontsize=15, fontweight='bold')

# Add cumulative line
cum_line = np.cumsum(sorted_values)
ax.plot(range(len(cum_line)), cum_line, 'k--o', markersize=5, linewidth=1.5, alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Leakage summary table
confidence_levels = {
    'order_failure': 'High',
    'sla_breach': 'Medium',
    'repeat_gap': 'Low (simulated)',
    'basket_gap': 'Medium',
    'freight': 'Medium',
    'seller_drag': 'Medium',
    'payment_friction': 'Medium',
}

pharma_priority = {
    'order_failure': 'Critical — Stockout prevention',
    'sla_breach': 'High — Same-day chronic Rx delivery',
    'repeat_gap': 'Critical — Auto-refill programs',
    'basket_gap': 'Medium — Pharmacist cross-sell prompts',
    'freight': 'High — Dark stores in Tier-2/3 cities',
    'seller_drag': 'Medium — Store ops intervention',
    'payment_friction': 'Medium — One-tap Rx reorder',
}

summary_data = []
for name, key in zip(layer_names, layer_keys):
    val = leakage.get(key, 0)
    summary_data.append({
        'Leakage Layer': name,
        'Amount (R$)': fmt_brl(val),
        '% of Total': f'{val/total_leakage*100:.1f}%' if total_leakage > 0 else '0%',
        'Confidence': confidence_levels.get(key, 'Low'),
        'Pharma Priority': pharma_priority.get(key, ''),
    })

summary_df = pd.DataFrame(summary_data)
print('REVENUE LEAKAGE SUMMARY')
print('=' * 110)
print(summary_df.to_string(index=False))
print('=' * 110)
print(f'TOTAL: {fmt_brl(total_leakage)} ({total_leakage/total_revenue*100:.1f}% of {fmt_brl(total_revenue)} total revenue)')

---
### Leakage Prediction Model

Can we predict **which orders are likely to leak** (cancel, get bad reviews) before they happen?

In [ ]:
# Leakage prediction model
# Target: is_leakage = canceled/unavailable OR delivered with review <= 2
model_df = df.copy()
model_df['is_leakage'] = (
    model_df['order_status'].isin(['canceled', 'unavailable']) |
    ((model_df['order_status'] == 'delivered') & (model_df['review_score'] <= 2))
).astype(int)

# Features
feature_cols = ['freight_ratio', 'payment_installments', 'price']

# Encode categoricals
le_payment = LabelEncoder()
model_df['payment_type_enc'] = le_payment.fit_transform(model_df['payment_type'].fillna('unknown'))
feature_cols.append('payment_type_enc')

le_state = LabelEncoder()
model_df['state_enc'] = le_state.fit_transform(model_df['customer_state'].fillna('unknown'))
feature_cols.append('state_enc')

# Seller quality score (merge from earlier)
if len(seller_metrics) > 0:
    model_df = model_df.merge(
        seller_metrics[['seller_id', 'composite_score']].rename(columns={'composite_score': 'seller_quality'}),
        on='seller_id', how='left'
    )
    model_df['seller_quality'] = model_df['seller_quality'].fillna(model_df['seller_quality'].median())
    feature_cols.append('seller_quality')

# Clean data for modeling
model_clean = model_df[feature_cols + ['is_leakage']].dropna()
X = model_clean[feature_cols]
y = model_clean['is_leakage']

print(f'Model dataset: {len(model_clean):,} rows')
print(f'Leakage rate: {y.mean()*100:.1f}%')
print(f'Features: {feature_cols}')

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: Logistic Regression (interpretable)
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_proba)

# Model 2: Gradient Boosting (accuracy)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                 random_state=42, subsample=0.8)
gb.fit(X_train, y_train)
gb_proba = gb.predict_proba(X_test)[:, 1]
gb_auc = roc_auc_score(y_test, gb_proba)

print(f'\nLogistic Regression AUC: {lr_auc:.3f}')
print(f'Gradient Boosting AUC:  {gb_auc:.3f}')

# Visualization: 2x2
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_proba)
fpr_gb, tpr_gb, _ = roc_curve(y_test, gb_proba)
axes[0, 0].plot(fpr_lr, tpr_lr, color=COLORS['blue'], linewidth=2, label=f'Logistic (AUC={lr_auc:.3f})')
axes[0, 0].plot(fpr_gb, tpr_gb, color=COLORS['green'], linewidth=2, label=f'GBM (AUC={gb_auc:.3f})')
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].set_title('ROC Curves', fontweight='bold')
axes[0, 0].legend(framealpha=0)

# 2. Feature importance (GBM)
importance = pd.Series(gb.feature_importances_, index=feature_cols).sort_values()
importance.plot(kind='barh', ax=axes[0, 1], color=COLORS['gold'], edgecolor='white')
axes[0, 1].set_title('Feature Importance (Gradient Boosting)', fontweight='bold')
axes[0, 1].set_xlabel('Importance')

# 3. Logistic Regression coefficients
coefs = pd.Series(lr.coef_[0], index=feature_cols).sort_values()
colors_coef = [COLORS['red'] if c > 0 else COLORS['green'] for c in coefs]
coefs.plot(kind='barh', ax=axes[1, 0], color=colors_coef, edgecolor='white')
axes[1, 0].set_title('Logistic Regression Coefficients', fontweight='bold')
axes[1, 0].set_xlabel('Coefficient (positive = increases leakage risk)')
axes[1, 0].axvline(0, color='black', linewidth=0.5)

# 4. Confusion matrix (GBM)
gb_pred = (gb_proba >= 0.5).astype(int)
cm = confusion_matrix(y_test, gb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[1, 1],
            xticklabels=['No Leak', 'Leak'], yticklabels=['No Leak', 'Leak'])
axes[1, 1].set_title('Confusion Matrix (Gradient Boosting)', fontweight='bold')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_xlabel('Predicted')

plt.suptitle('Leakage Prediction Model Performance', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

insights.add('Prediction', 'Leakage Risk Model',
             f'GBM model achieves AUC={gb_auc:.3f}. Top predictors: '
             f'{", ".join(importance.tail(3).index.tolist())}.',
             severity='opportunity')

> **Pharma Translation:** This model becomes a **real-time order risk flagging system**. When a Wellness Forever order has high-risk features (remote delivery zone, new customer, high-value Rx), the system should trigger proactive interventions: priority dispatch, SMS tracking, or pharmacist follow-up call. Even a modest AUC of 0.65+ is enough to prioritize the top 20% highest-risk orders for intervention.

In [ ]:
# Top 10 actionable triggers ranked by recoverable R$
triggers = [
    {
        'Rank': 1,
        'Trigger': 'Single-purchase customers in replenishable categories',
        'Recoverable R$': fmt_brl(leakage.get('repeat_gap', 0)),
        'Intervention': 'Auto-refill program with SMS/email reminders',
        'Pharma Equivalent': 'Chronic Rx auto-refill + WhatsApp reminders',
    },
    {
        'Rank': 2,
        'Trigger': 'Orders with freight > 30% of product price',
        'Recoverable R$': fmt_brl(leakage.get('freight', 0)),
        'Intervention': 'Subsidize freight above threshold; regional fulfillment',
        'Pharma Equivalent': 'Dark stores in Tier-2/3 cities; free delivery for Rx',
    },
    {
        'Rank': 3,
        'Trigger': 'Canceled/unavailable orders',
        'Recoverable R$': fmt_brl(leakage.get('order_failure', 0)),
        'Intervention': 'Stock monitoring alerts; substitute suggestions',
        'Pharma Equivalent': 'Never-stockout policy for top 200 SKUs',
    },
    {
        'Rank': 4,
        'Trigger': 'Single-category buyers (no cross-sell)',
        'Recoverable R$': fmt_brl(leakage.get('basket_gap', 0)),
        'Intervention': 'Post-purchase recommendations; bundle offers',
        'Pharma Equivalent': 'Pharmacist Rx-to-OTC attach rate prompts',
    },
    {
        'Rank': 5,
        'Trigger': 'Late deliveries causing churn',
        'Recoverable R$': fmt_brl(leakage.get('sla_breach', 0)),
        'Intervention': 'SLA breach alerts; proactive customer outreach',
        'Pharma Equivalent': 'Same-day guaranteed delivery for chronic Rx',
    },
    {
        'Rank': 6,
        'Trigger': 'Revenue through bottom-quartile sellers',
        'Recoverable R$': fmt_brl(leakage.get('seller_drag', 0)),
        'Intervention': 'Seller performance scorecards; training/exit',
        'Pharma Equivalent': 'Store ops intervention for bottom-Q locations',
    },
    {
        'Rank': 7,
        'Trigger': 'Boleto payment abandonment',
        'Recoverable R$': fmt_brl(leakage.get('payment_friction', 0)),
        'Intervention': 'Instant payment incentives; reduce boleto share',
        'Pharma Equivalent': 'One-tap UPI reorder for chronic medications',
    },
    {
        'Rank': 8,
        'Trigger': 'Orders to high-freight states (RR, AP, AC)',
        'Recoverable R$': 'Subset of #2',
        'Intervention': 'Regional hub in North/Northeast',
        'Pharma Equivalent': 'Distribution partnerships in remote regions',
    },
    {
        'Rank': 9,
        'Trigger': 'Deliveries > 14 days late (1-star reviews)',
        'Recoverable R$': 'Subset of #5',
        'Intervention': 'Proactive refund/voucher at day 7',
        'Pharma Equivalent': 'Auto-escalation for Rx delayed > 24 hours',
    },
    {
        'Rank': 10,
        'Trigger': 'High-installment credit card orders (13-24x)',
        'Recoverable R$': 'Subset of #7',
        'Intervention': 'Simplify checkout; pre-approved credit limits',
        'Pharma Equivalent': 'Health insurance co-pay integration',
    },
]

triggers_df = pd.DataFrame(triggers)
print('TOP 10 ACTIONABLE TRIGGERS')
print('=' * 130)
print(triggers_df.to_string(index=False))
print('=' * 130)

---
## Executive Closing

### Olist → Wellness Forever Mapping

| Olist Metric | Olist Finding | Wellness Forever Equivalent | Expected Impact |
|-------------|---------------|----------------------------|------------------|
| Order failure rate | ~1-2% of orders canceled | Rx fill-rate gaps | 5-10x higher per-order value in pharma |
| SLA breach → churn | Late orders get 3x more 1-star reviews | Late medicine = permanent switch | Chronic patients = R$50K+ LTV |
| Repeat rate | ~3% (marketplace model) | Chronic Rx refill rate target: 60-80% | Auto-refill = highest ROI intervention |
| Cross-sell gap | Multi-cat buyers spend 2x more | Rx-to-OTC attach rate | Pharmacist prompt converts 15-25% |
| Freight cost | >30% freight ratio kills conversion | Tier-2/3 delivery cost | Dark stores reduce last-mile 40-60% |
| Seller quality | Bottom-Q sellers = 12% revenue risk | Underperforming stores | Ops intervention team deployment |
| Payment friction | Boleto has higher failure rate | UPI/wallet checkout friction | One-tap reorder for chronic Rx |

### Next Step

**Share 6 months of anonymized order data — we'll run this exact analysis calibrated to Wellness Forever in 48 hours.**

What we need:
- Order table (order_id, date, status, customer_id)
- Line items (product, category, price, quantity)
- Delivery data (promised date, actual date)
- Payment data (method, amount, installments)
- Optional: store/location, customer region, prescription flag

In [ ]:
# Final insight summary
print(insights.summary())
print(f'\n{"=" * 70}')
print(f'TOTAL RECOVERABLE REVENUE LEAKAGE: {fmt_brl(total_leakage)}')
print(f'({total_leakage/total_revenue*100:.1f}% of {fmt_brl(total_revenue)} total revenue)')
print(f'{"=" * 70}')